# 阶段三：模型评估与优化

**实验目标：**
1. 超参调优（GridSearchCV）
2. 模型评估与可视化
3. 特征重要性分析
4. 业务洞察提炼

In [ ]:
# 环境初始化
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

from src.data_loader import load_processed_data
from src.model_utils import prepare_model_data
from src.evaluation import (
    calculate_metrics, plot_model_comparison,
    plot_prediction_vs_actual, plot_feature_importance, plot_residual_analysis
)

# 设置目录
DATA_DIR = Path('..') / 'data'
MODEL_DIR = Path('..') / 'models'
FIG_DIR = Path('..') / 'figures' / 'evaluation'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('✓ 环境初始化完成')

## 1. 加载数据并准备建模

In [ ]:
# 加载数据
house = load_processed_data(DATA_DIR)
X_train, X_test, y_train, y_test, scaler, feature_cols = prepare_model_data(house)

## 2. 超参调优

In [ ]:
# ========== 随机森林超参调优 ==========
print('=' * 60)
print('随机森林超参调优')
print('=' * 60)

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print(f'搜索空间：共 {3*4*3*3} 种参数组合')
print(f'总训练次数：{3*4*3*3 * 5} 次（5折交叉验证）')

rf_grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=rf_param_grid,
    scoring='r2', cv=5, n_jobs=-1, verbose=1, return_train_score=True
)
rf_grid.fit(X_train, y_train)

print(f'\n最优参数：{rf_grid.best_params_}')
print(f'最优交叉验证R2：{rf_grid.best_score_:.4f}')

In [ ]:
# ========== 梯度提升超参调优 ==========
print('=' * 60)
print('梯度提升超参调优')
print('=' * 60)

gb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

print(f'搜索空间：共 {3*3*3*2} 种参数组合')

gb_grid = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gb_param_grid,
    scoring='r2', cv=5, n_jobs=-1, verbose=1, return_train_score=True
)
gb_grid.fit(X_train, y_train)

print(f'\n最优参数：{gb_grid.best_params_}')
print(f'最优交叉验证R2：{gb_grid.best_score_:.4f}')

## 3. 调优后模型评估

In [ ]:
# 使用调优后的模型进行预测
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_pred_gb = best_gb.predict(X_test)

# 计算评估指标
results = [
    calculate_metrics(y_test, y_pred_rf, '随机森林（调优后）'),
    calculate_metrics(y_test, y_pred_gb, '梯度提升（调优后）')
]

results_df = pd.DataFrame(results)
print('\n' + '=' * 60)
print('调优后模型性能对比')
print('=' * 60)
display(results_df)

## 4. 模型评估可视化

In [ ]:
# 绘制预测vs真实值图
plot_prediction_vs_actual(y_test, y_pred_rf, '随机森林（调优后）', FIG_DIR)
plot_prediction_vs_actual(y_test, y_pred_gb, '梯度提升（调优后）', FIG_DIR)

In [ ]:
# 残差分析
plot_residual_analysis(y_test, y_pred_rf, '随机森林（调优后）', FIG_DIR)
plot_residual_analysis(y_test, y_pred_gb, '梯度提升（调优后）', FIG_DIR)

## 5. 特征重要性分析

In [ ]:
# 特征重要性
rf_importance = plot_feature_importance(best_rf, feature_cols, '随机森林（调优后）', FIG_DIR, top_n=15)
gb_importance = plot_feature_importance(best_gb, feature_cols, '梯度提升（调优后）', FIG_DIR, top_n=15)

In [ ]:
# 特征重要性对比
if rf_importance is not None and gb_importance is not None:
    importance_compare = pd.merge(
        rf_importance.rename(columns={'重要性': '随机森林'}),
        gb_importance.rename(columns={'重要性': '梯度提升'}),
        on='特征', how='outer'
    ).fillna(0)
    
    print('特征重要性对比（Top 10）：')
    display(importance_compare.head(10))

## 6. 保存最佳模型

In [ ]:
# 保存最佳模型
joblib.dump(best_rf, MODEL_DIR / 'best_rf_model.pkl')
joblib.dump(best_gb, MODEL_DIR / 'best_gb_model.pkl')
joblib.dump(scaler, MODEL_DIR / 'scaler.pkl')

print('✓ 最佳模型已保存')

## 阶段三评估总结

**完成的工作：**
1. 使用GridSearchCV对随机森林和梯度提升进行了超参调优
2. 对调优后的模型进行了全面评估
3. 完成了预测vs真实值可视化、残差分析
4. 分析了特征重要性

**主要发现：**
- 建筑面积、总价、单价等是影响房价的关键因素
- 学校等级评分、距离学校等学区相关特征对房价有显著影响
- 房龄、楼层等建筑特征也会影响房价